In [ ]:
!pip install -q "chronos-forecasting>=2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 3.6 MB/s eta 0:00:00


In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline

print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

GPU available: True Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

In [ ]:
# 1,000 test series, each 120 minutes of history (synthetic random walks, just for timing)
n, L = 1000, 120
rng = np.random.default_rng(0)
values = rng.normal(0, 1, (n, L)).cumsum(axis=1)
ts = pd.date_range("2020-01-01", periods=L, freq="min")
context = pd.DataFrame({"item_id": np.repeat(np.arange(n), L),
                        "timestamp": np.tile(ts, n),
                        "target": values.ravel()})

t0 = time.time()
pred = pipe.predict_df(context, prediction_length=5, quantile_levels=[0.1, 0.5, 0.9],
                       id_column="item_id", timestamp_column="timestamp", target="target")
dt = time.time() - t0

print(f"{n} forecasts in {dt:.1f} s → {n / dt:.0f} forecasts per second")
print(pred.columns.tolist())
pred.head(10)

1000 forecasts in 2.5 s → 398 forecasts per second
['item_id', 'timestamp', 'target_name', 'predictions', '0.1', '0.5', '0.9']


,item_id,timestamp,target_name,predictions,0.1,0.5,0.9
0,0,2020-01-01 02:00:00,target,9.938345,8.634457,9.938345,11.309607
1,0,2020-01-01 02:01:00,target,9.904993,7.990386,9.904993,11.772240
2,0,2020-01-01 02:02:00,target,9.791275,7.480876,9.791275,12.086277
3,0,2020-01-01 02:03:00,target,9.597383,6.843446,9.597383,12.290182
4,0,2020-01-01 02:04:00,target,9.442535,6.398743,9.442535,12.462337
5,1,2020-01-01 02:00:00,target,-14.628119,-16.482220,-14.628119,-13.051697
6,1,2020-01-01 02:01:00,target,-14.683102,-17.250505,-14.683102,-11.882023
7,1,2020-01-01 02:02:00,target,-14.676872,-17.861807,-14.676872,-11.589342
8,1,2020-01-01 02:03:00,target,-14.785112,-18.411766,-14.785112,-11.061028
9,1,2020-01-01 02:04:00,target,-14.739527,-18.893848,-14.739527,-11.111130


In [ ]:
import os
import time
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/assetmind"
OUTDIR = f"{BASE}/fm_scores"
os.makedirs(OUTDIR, exist_ok=True)

C, H, CHUNK = 120, 5, 20000          # 2 h context, 5 min ahead, forecasts per batch
SENSORS = ["P-TPT", "T-TPT", "P-ANULAR", "P-MON-CKP"]
m = pd.read_parquet(f"{BASE}/3w_means_ext.parquet")
print(m.shape)

Mounted at /content/drive
(475682, 6)


In [ ]:
series, ends, idx = {}, {}, []
span = C + H
for inst, g in m.groupby("instance_id", sort=False):
    ends[inst] = g["window_end"].to_numpy()
    for s in SENSORS:
        v = g[f"{s}_mean"].to_numpy(dtype=np.float32)
        series[(inst, s)] = v
        if len(v) < span:
            continue
        bad = np.concatenate([[0], np.cumsum(np.isnan(v))])
        clean = (bad[span:] - bad[:-span]) == 0          # no missing value in the 125-minute stretch
        positions = np.nonzero(clean)[0] + span - 1
        idx.extend((inst, s, int(i)) for i in positions)

print(f"{len(idx):,} forecasts to make → about {len(idx) / 429 / 60:.0f} min at 429/s")

726,915 forecasts to make → about 28 min at 429/s


In [ ]:
ts = pd.date_range("2000-01-01", periods=C, freq="min")   # placeholder timestamps (regular minutes)
t_start = time.time()

for c0 in range(0, len(idx), CHUNK):
    out_path = f"{OUTDIR}/chunk_{c0 // CHUNK:04d}.parquet"
    if os.path.exists(out_path):
        continue
    batch = idx[c0:c0 + CHUNK]
    k = len(batch)
    ctx = np.stack([series[(inst, s)][i - H - C + 1:i - H + 1] for inst, s, i in batch])
    act = np.stack([series[(inst, s)][i - H + 1:i + 1] for inst, s, i in batch])

    df = pd.DataFrame({"item_id": np.repeat(np.arange(k), C),
                       "timestamp": np.tile(ts, k),
                       "target": ctx.ravel()})
    pred = pipe.predict_df(df, prediction_length=H, quantile_levels=[0.1, 0.5, 0.9],
                           id_column="item_id", timestamp_column="timestamp", target="target")
    pred = pred.sort_values(["item_id", "timestamp"])
    q10, q50, q90 = (pred[q].to_numpy().reshape(k, H) for q in ["0.1", "0.5", "0.9"])

    width = np.maximum(q90 - q10, 1e-6).mean(axis=1)
    fmerr = np.abs(act - q50).mean(axis=1) / width
    fmout = (np.maximum(0, act - q90) + np.maximum(0, q10 - act)).mean(axis=1) / width

    pd.DataFrame({"instance_id": [b[0] for b in batch],
                  "sensor": [b[1] for b in batch],
                  "window_end": [ends[b[0]][b[2]] for b in batch],
                  "fmerr": fmerr, "fmout": fmout}).to_parquet(out_path, index=False)

    done = min(c0 + CHUNK, len(idx))
    print(f"{done:,}/{len(idx):,} done, {(time.time() - t_start) / 60:.1f} min elapsed")

In [ ]:
import glob
fm = pd.concat(pd.read_parquet(f) for f in sorted(glob.glob(f"{OUTDIR}/chunk_*.parquet")))
fm.to_parquet(f"{BASE}/3w_fm_scores.parquet", index=False)
print(fm.shape)
fm.describe()

(726915, 5)


,window_end,fmerr,fmout
count,726915,726915.000000,726915.000000
mean,2018-02-03 20:53:32.267729152,0.402293,0.121734
min,2011-08-31 03:02:20,0.000000,0.000000
25%,2017-02-08 07:32:59,0.167232,0.000000
50%,2018-08-05 14:35:37,0.283609,0.000000
75%,2018-10-28 01:37:26,0.457233,0.064541
max,2023-09-26 14:01:05,3665.753174,3665.229980
std,NaN,7.532431,7.528895


In [ ]:
curves = pd.concat([curves, pd.read_parquet(R / "3w_sweep_v10.parquet").assign(version=lambda d: d["config"])])
keep = ["v4", "v6_ext", "v7_ext_iforest", "v8_ext_ensemble", "v9_lstm_seed7", "v9_lstm_seed123",
        "v10a_fm_only", "v10b_v4_plus_fm"]
px.line(curves[curves["version"].isin(keep)].sort_values("false_alarms_per_day"),
        x="false_alarms_per_day", y="detection_macro", color="version", markers=True,
        title="Arm A vs Arm B: Chronos-2 zero-shot on the 3W scoreboard").show()